# 작물 질병 이미지 사전 전처리

**판다넘 팀 - 환경 통합 작물 질병 진행단계 분류**

이 노트북은 AI Hub 원본 데이터를 학습용 데이터셋으로 변환합니다.

## 처리 흐름

```
원본 이미지 (3024×3024)
  ↓ JSON 라벨 파싱 (bbox, risk, original)
  ↓ 원본 ID 기준 70:15:15 분할 (같은 원본의 증강은 같은 split)
  ↓ train만 그룹×클래스 캡 5000 적용
  ↓ bbox 크롭 + 30% 패딩
  ↓ Letterbox resize 512×512
  ↓ JPEG 저장 (품질 90) + manifest.csv
  ↓ 작물별 zip 압축
```

## 시작 전 체크리스트

- [ ] `config.py` 의 경로를 자기 환경에 맞게 수정했나요?
- [ ] `deeplearning_labeling.zip` 받았나요?
- [ ] 자기 담당 작물의 원본 zip을 `raw_data/{facility|outdoor}/{작물폴더}/{Training|Validation}/` 에 넣었나요?
- [ ] `pip install -r requirements.txt` 실행했나요?

> 환경 설정에 문제가 있으면 `AI_PROMPT_TEMPLATE.md` 참고해서 AI에게 도움 받으세요.

---
## 1. 환경 설정 및 모듈 로드

In [ ]:
import csv
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# 헬퍼 모듈 (같은 폴더의 .py 파일들)
from config import (
    RAW_DATA_ROOT, LABEL_ZIP_PATH, OUTPUT_ROOT,
    ENV_EN, SUBFOLDER_TRAIN, SUBFOLDER_VAL,
    MAX_PER_CLASS, TARGET_SIZE, JPEG_QUALITY, PADDING_RATIO,
    SPLIT_RATIO, RANDOM_SEED, RISK_NAMES,
    TRAIN_GROUPS, HELDOUT_GROUPS,
    group_id, group_type_of,
)
from label_utils    import LabelInfo, load_group_labels, summarize_labels
from split_utils    import assign_splits, split_distribution
from sampling_utils import apply_cap, cap_summary
from crop_utils     import preprocess_image

# 진행률 표시 (tqdm 없으면 그냥 통과)
try:
    from tqdm.notebook import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

print('모듈 로드 완료')

## 2. 경로 확인

`config.py` 의 경로 설정이 자기 환경에 맞는지 확인하세요.  
**문제 있으면 `config.py` 수정 후 위 셀부터 다시 실행하세요.**

In [ ]:
print(f'RAW_DATA_ROOT : {RAW_DATA_ROOT.resolve()}  존재: {RAW_DATA_ROOT.exists()}')
print(f'LABEL_ZIP_PATH: {LABEL_ZIP_PATH.resolve()}  존재: {LABEL_ZIP_PATH.exists()}')
print(f'OUTPUT_ROOT   : {OUTPUT_ROOT.resolve()}  (없으면 생성됨)')

print()
print(f'표준 설정 (절대 수정 금지):')
print(f'  RANDOM_SEED  : {RANDOM_SEED}')
print(f'  TARGET_SIZE  : {TARGET_SIZE}')
print(f'  JPEG_QUALITY : {JPEG_QUALITY}')
print(f'  PADDING_RATIO: {PADDING_RATIO}')
print(f'  MAX_PER_CLASS: {MAX_PER_CLASS}')
print(f'  SPLIT_RATIO  : {SPLIT_RATIO}')

assert LABEL_ZIP_PATH.exists(), '❌ deeplearning_labeling.zip 경로를 config.py 에서 수정하세요.'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('\n✓ 경로 검증 통과')

## 3. 자기 담당 그룹 선택

### 팀 분담표

| 담당 | 작물 |
|---|---|
| **이우현** | 시설 05.상추, 03.단호박, 11.토마토 |
| **장순규** | 시설 02.고추, 04.딸기, 09.쥬키니호박, 노지 07.잎마름병토마토* |
| **문재원** | 노지 03.배추, 01.고추, 시설 10.참외*, 노지 10.호박* |
| **박준환** | 노지 04.애호박, 06.오이, 시설 01.가지* |
| **신현준** | 노지 08.콩, 09.파, 시설 06.수박*, 노지 05.양배추* |

> `*` = held-out 그룹

### 자기 담당 입력

아래 셀에서 자기 담당 작물들을 `MY_GROUPS` 에 입력. 형식: `("시설"|"노지", "작물폴더명")`

**전체 작물 목록 (참고):**
- 학습 12그룹:
  - 시설: 02.고추, 03.단호박, 04.딸기, 05.상추, 09.쥬키니호박, 11.토마토
  - 노지: 01.고추, 03.배추, 04.애호박, 06.오이, 08.콩, 09.파
- held-out 6그룹:
  - 시설: 01.가지, 06.수박, 10.참외
  - 노지: 05.양배추, 07.잎마름병(토마토), 10.호박

In [ ]:
# ===== 자기가 처리할 그룹들 =====
MY_GROUPS = [
    ('시설', '02.고추'),
    # ('시설', '03.단호박'),
    # ('노지', '09.파'),
]
# ================================

print(f'처리 대상: {len(MY_GROUPS)}그룹')
for env, crop in MY_GROUPS:
    gid = group_id(env, crop)
    gtype = group_type_of(env, crop)
    print(f'  - {env} {crop} → {gid}  [{gtype}]')
    assert gtype != 'excluded', f'❌ ({env}, {crop}) 는 제외 그룹입니다.'
    
    # 원본 zip 폴더 존재 확인
    src = RAW_DATA_ROOT / ENV_EN[env] / crop
    train_dir = src / SUBFOLDER_TRAIN
    val_dir   = src / SUBFOLDER_VAL
    train_ok = train_dir.exists() and len(list(train_dir.glob('*.zip'))) > 0
    val_ok   = val_dir.exists()   and len(list(val_dir.glob('*.zip'))) > 0
    print(f'      Training: {"✓" if train_ok else "✗"} ({train_dir})')
    print(f'      Validation: {"✓" if val_ok else "✗"} ({val_dir})')

---
## 4. 그룹 단위 처리

각 그룹에 대해 다음 단계를 순차 진행합니다:

1. **라벨 로드**: `deeplearning_labeling.zip` 에서 JSON 파싱
2. **D-1 분할**: 원본 ID 기준 70:15:15 (같은 원본의 증강은 같은 split)
3. **캡 5000 적용**: train만 그룹×클래스별 상한
4. **이미지 전처리**: bbox 크롭 + 30% 패딩 + Letterbox resize 512×512 + JPEG 저장
5. **zip 압축**: 결과를 작물별 zip으로 묶기

아래 셀 함수들을 정의한 뒤, 마지막에 한 번에 실행합니다.

### 4.1 원본 zip 탐색 함수

In [ ]:
def find_source_zips(env: str, crop_folder: str) -> Dict[str, List[Path]]:
    """원본 zip 경로 수집."""
    base = RAW_DATA_ROOT / ENV_EN[env] / crop_folder
    result = {'train': [], 'val': []}
    for sub, key in [(SUBFOLDER_TRAIN, 'train'), (SUBFOLDER_VAL, 'val')]:
        sub_path = base / sub
        if sub_path.exists():
            result[key] = sorted(sub_path.glob('*.zip'))
    return result

### 4.2 그룹 처리 메인 함수

한 그룹의 전체 처리 흐름을 담은 함수입니다.

In [ ]:
def process_group(env: str, crop_folder: str) -> Optional[Dict]:
    """
    한 그룹(작물×환경) 전처리. 진행 상황 마크다운으로 표시.
    """
    gtype = group_type_of(env, crop_folder)
    gid   = group_id(env, crop_folder)

    print('\n' + '='*70)
    print(f'▶ [{gtype}] {env} {crop_folder}  →  {gid}')
    print('='*70)
    t0 = time.time()

    # ── 1. 라벨 로드 ──
    print('\n[1/5] 라벨 로드 중...')
    labels = load_group_labels(LABEL_ZIP_PATH, env, crop_folder)
    if not labels:
        print('  ❌ 라벨 없음. 종료.')
        return None
    summary = summarize_labels(labels)
    print(f"  → 총 {summary['total']:,}장 (원본 {summary['is_aug_false']:,} + 증강 {summary['is_aug_true']:,})")
    print(f"  risk 분포: { {RISK_NAMES[k]: v for k, v in sorted(summary['by_risk'].items())} }")
    print(f"  AI Hub split: {summary['by_split_aihub']}")

    # ── 2. D-1 분할 ──
    print('\n[2/5] D-1 분할 (원본 ID 기준 70:15:15)...')
    splits = assign_splits(labels, gtype)
    dist = split_distribution(splits)
    for split, info in dist.items():
        risks = {RISK_NAMES[k]: v for k, v in sorted(info['by_risk'].items())}
        print(f"  {split:18s}: {info['total']:>6,}장 (원본 {info['unique_originals']:>5,})  risk별 {risks}")

    # ── 3. 캡 적용 ──
    print(f'\n[3/5] 캡 {MAX_PER_CLASS} 적용 (train만)...')
    capped = apply_cap(splits) if gtype == 'train_group' else splits
    after_dist = split_distribution(capped)
    for split, info in after_dist.items():
        risks = {RISK_NAMES[k]: v for k, v in sorted(info['by_risk'].items())}
        print(f"  {split:18s}: {info['total']:>6,}장  risk별 {risks}")

    # ── 4. 원본 zip 탐색 ──
    print('\n[4/5] 원본 zip 탐색...')
    src_zips = find_source_zips(env, crop_folder)
    n_zips = sum(len(v) for v in src_zips.values())
    if n_zips == 0:
        print(f'  ❌ {RAW_DATA_ROOT}/{ENV_EN[env]}/{crop_folder}/ 에 zip 없음.')
        return None
    for k, zips in src_zips.items():
        print(f'  {k}: {len(zips)}개')
        for z in zips:
            size_gb = z.stat().st_size / (1024**3)
            print(f'    - {z.name} ({size_gb:.2f} GB)')

    # ── 5. 이미지 전처리 ──
    print('\n[5/5] 이미지 전처리 + 저장...')
    out_dir = OUTPUT_ROOT / gid
    img_dir = out_dir / 'images'
    img_dir.mkdir(parents=True, exist_ok=True)

    lookup = {l.image_filename: (l, s) for l, s in capped}
    manifest_rows = []
    counter = 0
    skipped_no_label = 0
    skipped_decode = 0

    for split_aihub, zip_list in src_zips.items():
        for zip_path in zip_list:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                entries = [e for e in zf.infolist()
                           if Path(e.filename).name.lower().endswith(('.jpg', '.jpeg', '.png'))]
                for entry in tqdm(entries, desc=zip_path.name, leave=False):
                    name = Path(entry.filename).name
                    if name not in lookup:
                        skipped_no_label += 1
                        continue
                    info, split = lookup[name]
                    with zf.open(entry) as f:
                        img_bytes = f.read()
                    processed = preprocess_image(img_bytes, info.bbox)
                    if processed is None:
                        skipped_decode += 1
                        continue
                    counter += 1
                    out_name = f'img_{counter:07d}.jpg'
                    with open(img_dir / out_name, 'wb') as f:
                        f.write(processed)
                    manifest_rows.append({
                        'file':         out_name,
                        'env':          info.env,
                        'crop_folder':  info.crop_folder,
                        'crop_code':    info.crop_code,
                        'disease':      info.disease,
                        'risk':         info.risk,
                        'grow':         info.grow,
                        'original_id':  info.original_id,
                        'is_aug':       info.is_aug,
                        'split':        split,
                        'group_type':   gtype,
                        'group_id':     gid,
                    })

    # manifest 저장
    if manifest_rows:
        keys = list(manifest_rows[0].keys())
        with open(out_dir / 'manifest.csv', 'w', encoding='utf-8-sig', newline='') as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(manifest_rows)

    elapsed = time.time() - t0
    print(f'\n  ✓ {counter:,}장 저장 (스킵: 라벨없음 {skipped_no_label:,} / 디코드 {skipped_decode:,})')
    print(f'  소요: {elapsed/60:.1f}분')
    print(f'  위치: {out_dir}')
    
    return {
        'group_id': gid, 'env': env, 'crop': crop_folder, 'group_type': gtype,
        'n_images': counter, 'n_skipped_no_label': skipped_no_label,
        'n_skipped_decode': skipped_decode, 'elapsed_min': elapsed/60,
    }

### 4.3 zip 압축 함수

In [ ]:
def zip_group_result(env: str, crop_folder: str) -> Optional[Path]:
    """전처리 결과를 zip으로 묶기."""
    gid = group_id(env, crop_folder)
    src = OUTPUT_ROOT / gid
    if not src.exists():
        return None
    zip_out = OUTPUT_ROOT / f'{gid}.zip'
    print(f'\nzip 압축: {zip_out.name} ...', end='', flush=True)
    t0 = time.time()
    with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_STORED) as zf:
        for p in src.rglob('*'):
            if p.is_file():
                zf.write(p, p.relative_to(src.parent))
    size_mb = zip_out.stat().st_size / (1024 ** 2)
    print(f' 완료 ({time.time()-t0:.1f}s, {size_mb:.1f} MB)')
    return zip_out

---
## 5. 실행

이제 자기 담당 그룹들을 처리합니다.

**작물 1개당 약 30~90분 소요** (CPU 코어 수, 디스크 속도에 따라).

> 셀 실행 중 멈춰도 같은 셀 다시 실행하면 처음부터 다시 됩니다 (덮어쓰기). 부분 재개는 안 됩니다.

In [ ]:
results = []
for env, crop in MY_GROUPS:
    result = process_group(env, crop)
    if result:
        results.append(result)
        zip_group_result(env, crop)

print('\n\n' + '='*70)
print(f'전체 완료: {len(results)}/{len(MY_GROUPS)} 그룹 처리됨')
print('='*70)
for r in results:
    print(f"  {r['group_id']:30s} {r['n_images']:>7,}장  ({r['elapsed_min']:.1f}분)")

---
## 6. 결과 검증

처리한 작물 중 하나의 manifest를 열어 결과를 확인합니다.

In [ ]:
import pandas as pd

if results:
    sample_gid = results[0]['group_id']
    manifest_path = OUTPUT_ROOT / sample_gid / 'manifest.csv'
    df = pd.read_csv(manifest_path)
    
    print(f'=== {sample_gid} manifest 검증 ===\n')
    print(f'전체 행: {len(df):,}')
    print(f'\nsplit 분포:')
    print(df['split'].value_counts())
    print(f'\nrisk 분포 (train만):')
    print(df[df['split']=='train']['risk'].value_counts().sort_index())
    print(f'\n원본 ID 중복 검사 (같은 ID가 여러 split에 있는지)...')
    multi_split = df.groupby('original_id')['split'].nunique()
    leaked = multi_split[multi_split > 1]
    if len(leaked) > 0:
        print(f'  ❌ 누수 발견: {len(leaked)}개 원본')
    else:
        print('  ✓ 누수 없음')
    print(f'\n첫 5행:')
    print(df.head())
else:
    print('아직 처리한 결과가 없습니다.')

---
## 7. 결과 전달

처리 끝난 zip을 팀장(또는 통합 담당자)에게 전달합니다:

```
preprocessed/
├── {group_id_1}.zip   ← 이걸 공유
├── {group_id_2}.zip
└── ...
```

구글 드라이브의 팀 공유 폴더에 업로드하면 됩니다.

**zip 파일에 다음이 포함되어 있는지 확인:**
- `images/` 폴더 (전처리된 JPEG들)
- `manifest.csv` (이미지별 메타데이터)

통합은 `merge_manifests.ipynb` 에서 한 명이 진행합니다.